# W2 Day2 (04/15 周二): CLIP + 对比学习

## 学习目标
- **理论 (1-1.5h)**: CLIP 双塔对比学习; InfoNCE / 温度参数; zero-shot / linear probe / fine-tune
- **实践 (2h)**: 手写 CLIP 框架 + 三种范式对比实验 (zero-shot vs linear probe vs fine-tune)
- **产出物**: CLIP 三范式对比 notebook

---

## 目录
1. [对比学习核心思想](#1)
2. [InfoNCE Loss 推导与实现](#2)
3. [温度参数的作用](#3)
4. [CLIP 架构详解](#4)
5. [手写 CLIP 完整实现](#5)
6. [CLIP 训练 (合成数据)](#6)
7. [范式一: Zero-Shot 分类](#7)
8. [范式二: Linear Probe](#8)
9. [范式三: Fine-Tune](#9)
10. [三范式对比分析](#10)
11. [使用 OpenAI 预训练 CLIP](#11)
12. [思考题 & 总结](#12)

In [ ]:
# 环境准备
# !pip install ftfy regex tqdm  # CLIP 依赖 (如果后面用 OpenAI CLIP)
# !pip install open_clip_torch  # 开源 CLIP 实现

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, Subset
import matplotlib.pyplot as plt
import numpy as np
import math
import time
from collections import defaultdict

torch.manual_seed(42)
np.random.seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
print(f"PyTorch: {torch.__version__}")

---
## 1. 对比学习核心思想 <a id='1'></a>

### 1.1 什么是对比学习？

对比学习的核心: **拉近正样本对 (positive pairs)，推远负样本对 (negative pairs)**

```
传统监督学习:  图片 → 模型 → "猫"  (需要标签)
对比学习:      图片A, 图片B → 相似/不相似  (不需要类别标签)
```

### 1.2 CLIP 的创新: 跨模态对比

CLIP 不是在同一模态内对比 (如 SimCLR)，而是:
- **正样本对**: 匹配的 (图片, 文本描述)
- **负样本对**: 不匹配的 (图片, 错误描述)

这使得模型能**理解图片和文本之间的语义对应关系**。

### 1.3 为什么 CLIP 如此重要？

1. **Zero-shot 能力**: 不需要在目标数据集上训练，就能分类
2. **通用表征**: 学到的特征可迁移到各种下游任务
3. **合成数据关联**: CLIP Score 是评估生成图片质量的标准指标 (→ W2 Day5, Day6)

---
## 2. InfoNCE Loss 推导与实现 <a id='2'></a>

### 2.1 从互信息到 InfoNCE

InfoNCE (Noise Contrastive Estimation) 是对比学习最常用的损失函数。

直觉: 对于 batch 中的 N 个正样本对 $(x_i, y_i)$:
- $x_i$ 应该和 $y_i$ 最相似 (正样本)
- $x_i$ 应该和 $y_j (j \neq i)$ 不相似 (负样本)

本质上就是一个 **N 分类问题**: 从 N 个候选中找到正确的配对。

### 2.2 公式

对于第 $i$ 个样本:
$$\mathcal{L}_i = -\log \frac{\exp(\text{sim}(x_i, y_i) / \tau)}{\sum_{j=1}^{N} \exp(\text{sim}(x_i, y_j) / \tau)}$$

其中:
- $\text{sim}(a, b) = \frac{a \cdot b}{\|a\| \|b\|}$ 是余弦相似度
- $\tau$ 是温度参数
- 分母是对所有 N 个候选求和

CLIP 使用**对称损失**: 同时从 image→text 和 text→image 两个方向计算:
$$\mathcal{L} = \frac{1}{2}(\mathcal{L}_{i2t} + \mathcal{L}_{t2i})$$

In [ ]:
def info_nce_loss(image_features, text_features, temperature=0.07):
    """
    手写 InfoNCE Loss (CLIP 的对比损失)
    
    Args:
        image_features: (N, D) L2 归一化的图像特征
        text_features:  (N, D) L2 归一化的文本特征
        temperature:    温度参数
    
    Returns:
        loss: 标量
    """
    N = image_features.size(0)
    
    # Step 1: 计算相似度矩阵 (N, N)
    # 因为已经 L2 归一化，点积 = 余弦相似度
    logits = image_features @ text_features.T / temperature  # (N, N)
    
    # Step 2: 目标标签 (对角线是正样本对)
    labels = torch.arange(N, device=logits.device)
    
    # Step 3: 对称 Cross-Entropy Loss
    loss_i2t = F.cross_entropy(logits, labels)       # image → text
    loss_t2i = F.cross_entropy(logits.T, labels)     # text → image
    
    loss = (loss_i2t + loss_t2i) / 2
    
    return loss, logits


# 验证实现
print("=" * 60)
print("InfoNCE Loss 验证")
print("=" * 60)

# 模拟: 完美对齐的特征 (正样本对相似度=1，负样本对=0)
N, D = 8, 64
perfect_img = F.normalize(torch.randn(N, D), dim=-1)
perfect_txt = perfect_img.clone()  # 完美匹配

loss_perfect, logits_perfect = info_nce_loss(perfect_img, perfect_txt, temperature=0.07)
print(f"完美对齐: loss = {loss_perfect.item():.4f} (理论下界 ≈ 0)")

# 随机特征 (无对齐)
random_img = F.normalize(torch.randn(N, D), dim=-1)
random_txt = F.normalize(torch.randn(N, D), dim=-1)
loss_random, _ = info_nce_loss(random_img, random_txt, temperature=0.07)
print(f"随机特征: loss = {loss_random.item():.4f} (理论上界 ≈ log(N) = {math.log(N):.4f})")

In [ ]:
# 可视化相似度矩阵
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# 完美对齐
sim_perfect = (perfect_img @ perfect_txt.T).detach().numpy()
im0 = axes[0].imshow(sim_perfect, cmap='RdBu_r', vmin=-1, vmax=1)
axes[0].set_title(f'完美对齐\nloss={loss_perfect.item():.4f}')
axes[0].set_xlabel('Text index')
axes[0].set_ylabel('Image index')
plt.colorbar(im0, ax=axes[0])

# 随机
sim_random = (random_img @ random_txt.T).detach().numpy()
im1 = axes[1].imshow(sim_random, cmap='RdBu_r', vmin=-1, vmax=1)
axes[1].set_title(f'随机特征\nloss={loss_random.item():.4f}')
axes[1].set_xlabel('Text index')
axes[1].set_ylabel('Image index')
plt.colorbar(im1, ax=axes[1])

plt.suptitle('InfoNCE: 相似度矩阵 (对角线应最亮)', fontsize=13)
plt.tight_layout()
plt.show()

---
## 3. 温度参数 $\tau$ 的作用 <a id='3'></a>

温度参数控制 softmax 分布的**尖锐程度**:

- $\tau \to 0$: 分布极度尖锐 → hard assignment, 但梯度消失
- $\tau \to \infty$: 分布均匀 → 所有样本都"差不多"，学不到东西
- 适中的 $\tau$: 平衡区分度和训练稳定性

CLIP 论文: $\tau$ 初始化为 $\exp(\log(1/0.07)) \approx 14.3$，作为**可学习参数**。

In [ ]:
# 实验: 温度参数对 loss 和梯度的影响
print("=" * 60)
print("实验: 温度参数 τ 的影响")
print("=" * 60)

# 半对齐的特征 (有一定噪声)
N, D = 32, 128
base = F.normalize(torch.randn(N, D), dim=-1)
img_feat = F.normalize(base + 0.3 * torch.randn(N, D), dim=-1)
txt_feat = F.normalize(base + 0.3 * torch.randn(N, D), dim=-1)

temperatures = [0.01, 0.03, 0.07, 0.1, 0.3, 0.5, 1.0, 2.0, 5.0]
losses = []
grad_norms = []
accuracies = []

for tau in temperatures:
    img_f = img_feat.clone().requires_grad_(True)
    txt_f = txt_feat.clone().detach()
    
    loss, logits = info_nce_loss(img_f, txt_f, temperature=tau)
    loss.backward()
    
    # 准确率: 对角线是否是最大值
    preds = logits.detach().argmax(dim=1)
    acc = (preds == torch.arange(N)).float().mean().item()
    
    losses.append(loss.item())
    grad_norms.append(img_f.grad.norm().item())
    accuracies.append(acc * 100)
    
    print(f"  τ={tau:5.2f} | loss={loss.item():8.4f} | grad_norm={img_f.grad.norm().item():8.4f} | acc={acc*100:.1f}%")

# 画图
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

axes[0].semilogx(temperatures, losses, 'bo-')
axes[0].set_xlabel('Temperature τ')
axes[0].set_ylabel('Loss')
axes[0].set_title('Loss vs Temperature')
axes[0].axvline(x=0.07, color='r', linestyle='--', alpha=0.5, label='τ=0.07 (CLIP)')
axes[0].legend()

axes[1].semilogx(temperatures, grad_norms, 'rs-')
axes[1].set_xlabel('Temperature τ')
axes[1].set_ylabel('Gradient Norm')
axes[1].set_title('Gradient Norm vs Temperature')

axes[2].semilogx(temperatures, accuracies, 'g^-')
axes[2].set_xlabel('Temperature τ')
axes[2].set_ylabel('Retrieval Accuracy (%)')
axes[2].set_title('Accuracy vs Temperature')

plt.tight_layout()
plt.show()

print("\n💡 τ 太小: loss 很大 (数值不稳定) / τ 太大: 分辨不了正负样本")
print("💡 CLIP 使用 τ=0.07 作为初始值，并让模型自己学习")

---
## 4. CLIP 架构详解 <a id='4'></a>

### 4.1 双塔结构

```
          Image               Text
           │                    │
    ┌──────▼──────┐     ┌──────▼──────┐
    │ Image       │     │ Text        │
    │ Encoder     │     │ Encoder     │
    │ (ViT/ResNet)│     │ (Transformer│
    └──────┬──────┘     │  Decoder)   │
           │            └──────┬──────┘
    ┌──────▼──────┐     ┌──────▼──────┐
    │ Projection  │     │ Projection  │
    │ Head        │     │ Head        │
    └──────┬──────┘     └──────┬──────┘
           │                    │
    ┌──────▼──────┐     ┌──────▼──────┐
    │ L2 Norm     │     │ L2 Norm     │
    └──────┬──────┘     └──────┬──────┘
           │                    │
           └───── cosine sim ───┘
                    │
              InfoNCE Loss
```

### 4.2 关键设计选择

1. **Image Encoder**: 可以是 ResNet 或 ViT
2. **Text Encoder**: Transformer (但只用 causal / decoder 模式)
3. **Projection Head**: 将两个模态映射到**同一维度**的共享空间
4. **L2 归一化**: 确保余弦相似度在 [-1, 1] 范围
5. **可学习温度**: 让模型自适应调整 softmax 的尖锐程度

---
## 5. 手写 CLIP 完整实现 <a id='5'></a>

我们在 CIFAR-10 上实现一个小型 CLIP，用类别名作为文本。

In [ ]:
# ==================== Image Encoder ====================
class SimpleImageEncoder(nn.Module):
    """
    简化版 Image Encoder (小型 CNN)
    实际 CLIP 使用 ViT-B/32 或 ResNet-50
    """
    def __init__(self, embed_dim=256):
        super().__init__()
        self.features = nn.Sequential(
            # Block 1: 32x32 -> 16x16
            nn.Conv2d(3, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.GELU(),
            nn.Conv2d(64, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.GELU(),
            nn.MaxPool2d(2),
            
            # Block 2: 16x16 -> 8x8
            nn.Conv2d(64, 128, 3, padding=1),
            nn.BatchNorm2d(128),
            nn.GELU(),
            nn.Conv2d(128, 128, 3, padding=1),
            nn.BatchNorm2d(128),
            nn.GELU(),
            nn.MaxPool2d(2),
            
            # Block 3: 8x8 -> 4x4
            nn.Conv2d(128, 256, 3, padding=1),
            nn.BatchNorm2d(256),
            nn.GELU(),
            nn.AdaptiveAvgPool2d(1),  # Global Average Pooling
        )
        self.projection = nn.Linear(256, embed_dim)
    
    def forward(self, x):
        """x: (batch, 3, 32, 32) -> (batch, embed_dim)"""
        x = self.features(x)
        x = x.flatten(1)  # (batch, 256)
        x = self.projection(x)  # (batch, embed_dim)
        return x


# ==================== Text Encoder ====================
class SimpleTextEncoder(nn.Module):
    """
    简化版 Text Encoder
    对于 CIFAR-10 这种简单场景，文本就是类别名
    我们用一个可学习的 token embedding + Transformer
    """
    def __init__(self, vocab_size, embed_dim=256, max_len=32, n_heads=4, n_layers=2):
        super().__init__()
        self.token_embedding = nn.Embedding(vocab_size, embed_dim)
        self.position_embedding = nn.Embedding(max_len, embed_dim)
        
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim, nhead=n_heads, dim_feedforward=embed_dim*4,
            dropout=0.1, activation='gelu', batch_first=True, norm_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)
        self.final_ln = nn.LayerNorm(embed_dim)
        self.projection = nn.Linear(embed_dim, embed_dim)
    
    def forward(self, tokens):
        """
        tokens: (batch, seq_len) token indices
        返回: (batch, embed_dim) 文本特征 (取 [EOS] 位置，类似 CLS)
        """
        B, T = tokens.shape
        pos = torch.arange(T, device=tokens.device).unsqueeze(0)
        
        x = self.token_embedding(tokens) + self.position_embedding(pos)
        
        # Causal mask (CLIP 的 text encoder 用 causal attention)
        causal_mask = torch.triu(
            torch.ones(T, T, device=tokens.device) * float('-inf'), diagonal=1
        )
        x = self.transformer(x, mask=causal_mask)
        
        # 取最后一个非 padding token 的特征
        # 简化: 取最后一个位置
        x = self.final_ln(x[:, -1, :])  # (batch, embed_dim)
        x = self.projection(x)
        return x


# ==================== CLIP Model ====================
class CLIP(nn.Module):
    """
    完整 CLIP 模型 (简化版)
    """
    def __init__(self, image_encoder, text_encoder, embed_dim=256):
        super().__init__()
        self.image_encoder = image_encoder
        self.text_encoder = text_encoder
        
        # 可学习温度参数 (CLIP 论文的设计)
        # 初始化为 log(1/0.07) ≈ 2.66
        self.logit_scale = nn.Parameter(torch.ones([]) * np.log(1 / 0.07))
    
    def encode_image(self, images):
        """编码图像并 L2 归一化"""
        features = self.image_encoder(images)
        return F.normalize(features, dim=-1)
    
    def encode_text(self, tokens):
        """编码文本并 L2 归一化"""
        features = self.text_encoder(tokens)
        return F.normalize(features, dim=-1)
    
    def forward(self, images, tokens):
        """
        计算 CLIP loss
        images: (N, 3, H, W)
        tokens: (N, seq_len)
        """
        image_features = self.encode_image(images)
        text_features = self.encode_text(tokens)
        
        # 可学习温度 (clamp 防止过大)
        logit_scale = self.logit_scale.exp().clamp(max=100)
        
        # 相似度矩阵
        logits_per_image = logit_scale * image_features @ text_features.T
        logits_per_text = logits_per_image.T
        
        # 对称 CE Loss
        labels = torch.arange(len(images), device=images.device)
        loss_i2t = F.cross_entropy(logits_per_image, labels)
        loss_t2i = F.cross_entropy(logits_per_text, labels)
        loss = (loss_i2t + loss_t2i) / 2
        
        return loss, logits_per_image, logits_per_text


# 测试
img_enc = SimpleImageEncoder(embed_dim=256)
txt_enc = SimpleTextEncoder(vocab_size=100, embed_dim=256)
clip_model = CLIP(img_enc, txt_enc, embed_dim=256)

dummy_imgs = torch.randn(4, 3, 32, 32)
dummy_toks = torch.randint(0, 100, (4, 8))
loss, _, _ = clip_model(dummy_imgs, dummy_toks)

total_params = sum(p.numel() for p in clip_model.parameters())
print(f"CLIP 模型参数量: {total_params:,} ({total_params/1e6:.1f}M)")
print(f"测试 loss: {loss.item():.4f}")
print(f"初始温度: τ = 1/exp(logit_scale) = {1/clip_model.logit_scale.exp().item():.4f}")

---
## 6. CLIP 训练 (CIFAR-10) <a id='6'></a>

我们用 CIFAR-10 训练 CLIP:
- 图片: CIFAR-10 的 32×32 图片
- 文本: 用模板 `"a photo of a {class_name}"` 生成
- 正样本对: (图片, 对应类别的文本)

### 6.1 文本 Tokenizer

实际 CLIP 使用 BPE tokenizer，这里简化为字符级 tokenizer。

In [ ]:
# 简单 tokenizer
class SimpleTokenizer:
    """字符级 tokenizer (教学用)"""
    def __init__(self):
        # 基本字符集
        chars = list(" abcdefghijklmnopqrstuvwxyz0123456789.,!?")
        self.char2idx = {c: i+2 for i, c in enumerate(chars)}  # 0=PAD, 1=EOS
        self.idx2char = {i: c for c, i in self.char2idx.items()}
        self.vocab_size = len(self.char2idx) + 2  # +PAD +EOS
    
    def encode(self, text, max_len=32):
        """将文本编码为 token 序列"""
        text = text.lower()
        tokens = [self.char2idx.get(c, 0) for c in text[:max_len-1]]
        tokens.append(1)  # EOS
        # Padding
        tokens += [0] * (max_len - len(tokens))
        return torch.tensor(tokens, dtype=torch.long)
    
    def batch_encode(self, texts, max_len=32):
        return torch.stack([self.encode(t, max_len) for t in texts])

tokenizer = SimpleTokenizer()

# CIFAR-10 类别名
CIFAR10_CLASSES = [
    'airplane', 'automobile', 'bird', 'cat', 'deer',
    'dog', 'frog', 'horse', 'ship', 'truck'
]

# 文本模板
TEMPLATES = [
    "a photo of a {}",
    "a picture of a {}",
    "an image of a {}",
    "a {} in a photo",
    "this is a {}",
]

def get_text_for_label(label_idx):
    """根据标签生成随机文本描述"""
    class_name = CIFAR10_CLASSES[label_idx]
    template = TEMPLATES[np.random.randint(len(TEMPLATES))]
    return template.format(class_name)

# 测试
for i in range(5):
    text = get_text_for_label(i)
    tokens = tokenizer.encode(text)
    print(f"Label {i} ({CIFAR10_CLASSES[i]}): '{text}' -> tokens[:15]={tokens[:15].tolist()}")

print(f"\nVocab size: {tokenizer.vocab_size}")

In [ ]:
# 加载 CIFAR-10
transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616)),
])

transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616)),
])

train_dataset = torchvision.datasets.CIFAR10(
    root='./data', train=True, download=True, transform=transform_train
)
test_dataset = torchvision.datasets.CIFAR10(
    root='./data', train=False, download=True, transform=transform_test
)

# 为了训练速度，使用子集
TRAIN_SIZE = 10000
TEST_SIZE = 2000
train_subset = Subset(train_dataset, range(TRAIN_SIZE))
test_subset = Subset(test_dataset, range(TEST_SIZE))

train_loader = DataLoader(train_subset, batch_size=128, shuffle=True, num_workers=2)
test_loader = DataLoader(test_subset, batch_size=256, shuffle=False, num_workers=2)

print(f"训练集: {len(train_subset)} 张图片")
print(f"测试集: {len(test_subset)} 张图片")

In [ ]:
# 训练 CLIP
def train_clip(n_epochs=20):
    # 创建模型
    img_enc = SimpleImageEncoder(embed_dim=256)
    txt_enc = SimpleTextEncoder(vocab_size=tokenizer.vocab_size, embed_dim=256)
    model = CLIP(img_enc, txt_enc, embed_dim=256).to(device)
    
    optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=0.01)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=n_epochs)
    
    train_losses = []
    temp_history = []
    
    for epoch in range(n_epochs):
        model.train()
        epoch_loss = 0
        n_batches = 0
        
        for images, labels in train_loader:
            images = images.to(device)
            
            # 为每个样本生成文本描述
            texts = [get_text_for_label(l.item()) for l in labels]
            tokens = tokenizer.batch_encode(texts).to(device)
            
            # 前向
            loss, _, _ = model(images, tokens)
            
            # 反向
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            
            epoch_loss += loss.item()
            n_batches += 1
        
        scheduler.step()
        avg_loss = epoch_loss / n_batches
        train_losses.append(avg_loss)
        
        tau = 1.0 / model.logit_scale.exp().item()
        temp_history.append(tau)
        
        if (epoch + 1) % 5 == 0 or epoch == 0:
            print(f"Epoch {epoch+1:3d} | Loss: {avg_loss:.4f} | τ: {tau:.4f} | "
                  f"logit_scale: {model.logit_scale.item():.2f}")
    
    return model, train_losses, temp_history

print("开始训练 CLIP...")
clip_model, train_losses, temp_history = train_clip(n_epochs=25)

In [ ]:
# 训练曲线
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(train_losses, 'b-', linewidth=2)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('InfoNCE Loss')
axes[0].set_title('CLIP Training Loss')

axes[1].plot(temp_history, 'r-', linewidth=2)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Temperature τ')
axes[1].set_title('学习到的温度参数变化')
axes[1].axhline(y=0.07, color='gray', linestyle='--', alpha=0.5, label='初始 τ=0.07')
axes[1].legend()

plt.tight_layout()
plt.show()

---
## 7. 范式一: Zero-Shot 分类 <a id='7'></a>

Zero-shot 的核心思想:
1. 对每个类别生成文本 prompt: `"a photo of a {class_name}"`
2. 编码所有类别文本 → N 个文本特征向量
3. 编码待分类图片 → 图像特征向量
4. 用余弦相似度找最接近的类别

**不需要任何训练数据！只需要类别名称。**

In [ ]:
@torch.no_grad()
def zero_shot_classify(model, images, class_names, templates=None):
    """
    CLIP Zero-Shot 分类
    
    Args:
        model: CLIP 模型
        images: (N, 3, H, W) 图片
        class_names: 类别名列表
        templates: 文本模板列表 (用于 prompt ensembling)
    
    Returns:
        predictions: (N,) 预测类别
        similarities: (N, C) 相似度矩阵
    """
    model.eval()
    
    if templates is None:
        templates = ["a photo of a {}"]
    
    # Step 1: 编码所有类别文本 (可以用多个模板做 ensemble)
    all_text_features = []
    for class_name in class_names:
        class_features = []
        for template in templates:
            text = template.format(class_name)
            tokens = tokenizer.encode(text).unsqueeze(0).to(images.device)
            feat = model.encode_text(tokens)
            class_features.append(feat)
        # 多模板平均
        avg_feature = torch.stack(class_features).mean(dim=0)
        avg_feature = F.normalize(avg_feature, dim=-1)
        all_text_features.append(avg_feature)
    
    text_features = torch.cat(all_text_features, dim=0)  # (C, D)
    
    # Step 2: 编码图片
    image_features = model.encode_image(images)  # (N, D)
    
    # Step 3: 计算相似度
    similarities = image_features @ text_features.T  # (N, C)
    
    # Step 4: 取最大相似度的类别
    predictions = similarities.argmax(dim=-1)
    
    return predictions, similarities


# Zero-Shot 评估
print("=" * 60)
print("范式一: Zero-Shot Classification")
print("=" * 60)

clip_model.eval()
all_preds = []
all_labels = []

for images, labels in test_loader:
    images = images.to(device)
    
    # 单模板
    preds, sims = zero_shot_classify(clip_model, images, CIFAR10_CLASSES)
    all_preds.append(preds.cpu())
    all_labels.append(labels)

all_preds = torch.cat(all_preds)
all_labels = torch.cat(all_labels)
zero_shot_acc_single = (all_preds == all_labels).float().mean().item() * 100

# 多模板 ensemble
all_preds_ensemble = []
for images, labels in test_loader:
    images = images.to(device)
    preds, _ = zero_shot_classify(clip_model, images, CIFAR10_CLASSES, templates=TEMPLATES)
    all_preds_ensemble.append(preds.cpu())

all_preds_ensemble = torch.cat(all_preds_ensemble)
zero_shot_acc_ensemble = (all_preds_ensemble == all_labels).float().mean().item() * 100

print(f"Zero-Shot (单模板) 准确率: {zero_shot_acc_single:.1f}%")
print(f"Zero-Shot (多模板 ensemble) 准确率: {zero_shot_acc_ensemble:.1f}%")
print(f"\n💡 Prompt Ensemble 通常能提升 2-5% 的准确率")

In [ ]:
# 可视化 Zero-Shot 分类结果
fig, axes = plt.subplots(2, 5, figsize=(16, 7))

# 反归一化用于可视化
mean = torch.tensor([0.4914, 0.4822, 0.4465]).view(3, 1, 1)
std = torch.tensor([0.2470, 0.2435, 0.2616]).view(3, 1, 1)

# 取几个样本
sample_images, sample_labels = next(iter(test_loader))
sample_images_gpu = sample_images[:10].to(device)
preds, sims = zero_shot_classify(clip_model, sample_images_gpu, CIFAR10_CLASSES, TEMPLATES)

for i in range(10):
    ax = axes[i // 5, i % 5]
    
    # 反归一化
    img = sample_images[i] * std + mean
    img = img.clamp(0, 1).permute(1, 2, 0).numpy()
    ax.imshow(img)
    
    true_label = CIFAR10_CLASSES[sample_labels[i]]
    pred_label = CIFAR10_CLASSES[preds[i]]
    color = 'green' if preds[i] == sample_labels[i] else 'red'
    
    ax.set_title(f'True: {true_label}\nPred: {pred_label}', color=color, fontsize=10)
    ax.axis('off')

plt.suptitle('Zero-Shot Classification Results (绿色=正确, 红色=错误)', fontsize=13)
plt.tight_layout()
plt.show()

---
## 8. 范式二: Linear Probe <a id='8'></a>

Linear Probe:
1. **冻结** CLIP 的 image encoder (不更新参数)
2. 在其输出上训练一个**线性分类头**
3. 衡量 CLIP 学到的特征质量

如果 linear probe 效果好 → 说明 CLIP 已经学到了很好的特征表征。

In [ ]:
@torch.no_grad()
def extract_features(model, dataloader):
    """
    用 CLIP image encoder 提取特征
    """
    model.eval()
    all_features = []
    all_labels = []
    
    for images, labels in dataloader:
        images = images.to(device)
        features = model.encode_image(images)  # 已经 L2 归一化
        all_features.append(features.cpu())
        all_labels.append(labels)
    
    return torch.cat(all_features), torch.cat(all_labels)


# 提取特征
print("提取 CLIP 图像特征...")
train_features, train_labels = extract_features(clip_model, train_loader)
test_features, test_labels = extract_features(clip_model, test_loader)
print(f"训练特征: {train_features.shape}")
print(f"测试特征: {test_features.shape}")

In [ ]:
# Linear Probe
print("\n" + "=" * 60)
print("范式二: Linear Probe")
print("=" * 60)

class LinearProbe(nn.Module):
    def __init__(self, input_dim, n_classes):
        super().__init__()
        self.linear = nn.Linear(input_dim, n_classes)
    
    def forward(self, x):
        return self.linear(x)

def train_linear_probe(train_features, train_labels, test_features, test_labels, n_epochs=50):
    probe = LinearProbe(train_features.shape[1], 10).to(device)
    optimizer = torch.optim.Adam(probe.parameters(), lr=1e-3)
    criterion = nn.CrossEntropyLoss()
    
    # 移到 GPU
    train_feat = train_features.to(device)
    train_lab = train_labels.to(device)
    test_feat = test_features.to(device)
    test_lab = test_labels.to(device)
    
    losses = []
    accs = []
    
    for epoch in range(n_epochs):
        probe.train()
        
        # 小数据集直接全量训练
        logits = probe(train_feat)
        loss = criterion(logits, train_lab)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        losses.append(loss.item())
        
        # 测试
        if (epoch + 1) % 10 == 0 or epoch == 0:
            probe.eval()
            with torch.no_grad():
                test_logits = probe(test_feat)
                test_preds = test_logits.argmax(dim=1)
                acc = (test_preds == test_lab).float().mean().item() * 100
                accs.append((epoch, acc))
                print(f"  Epoch {epoch+1:3d} | Loss: {loss.item():.4f} | Test Acc: {acc:.1f}%")
    
    # 最终准确率
    probe.eval()
    with torch.no_grad():
        test_logits = probe(test_feat)
        test_preds = test_logits.argmax(dim=1)
        final_acc = (test_preds == test_lab).float().mean().item() * 100
    
    return probe, final_acc, losses, accs


probe, linear_probe_acc, lp_losses, lp_accs = train_linear_probe(
    train_features, train_labels, test_features, test_labels
)
print(f"\nLinear Probe 最终准确率: {linear_probe_acc:.1f}%")

---
## 9. 范式三: Fine-Tune <a id='9'></a>

Fine-tune 整个 image encoder + 分类头:
- **解冻** CLIP 的 image encoder
- 使用较小的学习率更新所有参数
- 通常效果最好，但需要更多训练数据，且可能破坏预训练特征

In [ ]:
print("=" * 60)
print("范式三: Full Fine-Tune")
print("=" * 60)

class FineTuneModel(nn.Module):
    def __init__(self, clip_image_encoder, embed_dim, n_classes):
        super().__init__()
        # 复制 image encoder (不影响原始 CLIP 模型)
        self.encoder = copy.deepcopy(clip_image_encoder)
        self.classifier = nn.Linear(embed_dim, n_classes)
    
    def forward(self, x):
        # 注意: 不做 L2 归一化 (分类任务不需要)
        features = self.encoder(x)
        return self.classifier(features)


def train_finetune(clip_model, n_epochs=20):
    model = FineTuneModel(clip_model.image_encoder, 256, 10).to(device)
    
    # 差异化学习率: encoder 小，classifier 大
    optimizer = torch.optim.AdamW([
        {'params': model.encoder.parameters(), 'lr': 1e-4},
        {'params': model.classifier.parameters(), 'lr': 1e-3},
    ], weight_decay=0.01)
    
    criterion = nn.CrossEntropyLoss()
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=n_epochs)
    
    losses = []
    accs = []
    
    for epoch in range(n_epochs):
        model.train()
        epoch_loss = 0
        n_batches = 0
        
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            
            logits = model(images)
            loss = criterion(logits, labels)
            
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            
            epoch_loss += loss.item()
            n_batches += 1
        
        scheduler.step()
        avg_loss = epoch_loss / n_batches
        losses.append(avg_loss)
        
        if (epoch + 1) % 5 == 0 or epoch == 0:
            model.eval()
            correct = 0
            total = 0
            with torch.no_grad():
                for images, labels in test_loader:
                    images, labels = images.to(device), labels.to(device)
                    preds = model(images).argmax(dim=1)
                    correct += (preds == labels).sum().item()
                    total += len(labels)
            acc = correct / total * 100
            accs.append((epoch, acc))
            print(f"  Epoch {epoch+1:3d} | Loss: {avg_loss:.4f} | Test Acc: {acc:.1f}%")
    
    # 最终评估
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            preds = model(images).argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += len(labels)
    final_acc = correct / total * 100
    
    return model, final_acc, losses, accs


ft_model, finetune_acc, ft_losses, ft_accs = train_finetune(clip_model, n_epochs=20)
print(f"\nFine-Tune 最终准确率: {finetune_acc:.1f}%")

In [ ]:
# 额外对比: 从零训练 (Baseline)
print("\n" + "=" * 60)
print("Baseline: 从零训练 CNN (无 CLIP 预训练)")
print("=" * 60)

def train_from_scratch(n_epochs=20):
    # 使用相同架构但随机初始化
    model = nn.Sequential(
        SimpleImageEncoder(embed_dim=256),
        nn.Linear(256, 10)
    ).to(device)
    
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=0.01)
    criterion = nn.CrossEntropyLoss()
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=n_epochs)
    
    losses = []
    accs = []
    
    for epoch in range(n_epochs):
        model.train()
        epoch_loss = 0
        n_batches = 0
        
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            logits = model(images)
            loss = criterion(logits, labels)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()
            n_batches += 1
        
        scheduler.step()
        avg_loss = epoch_loss / n_batches
        losses.append(avg_loss)
        
        if (epoch + 1) % 5 == 0 or epoch == 0:
            model.eval()
            correct = 0
            total = 0
            with torch.no_grad():
                for images, labels in test_loader:
                    images, labels = images.to(device), labels.to(device)
                    preds = model(images).argmax(dim=1)
                    correct += (preds == labels).sum().item()
                    total += len(labels)
            acc = correct / total * 100
            accs.append((epoch, acc))
            print(f"  Epoch {epoch+1:3d} | Loss: {avg_loss:.4f} | Test Acc: {acc:.1f}%")
    
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            preds = model(images).argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += len(labels)
    final_acc = correct / total * 100
    return final_acc, losses, accs

scratch_acc, scratch_losses, scratch_accs = train_from_scratch(n_epochs=20)
print(f"\n从零训练 最终准确率: {scratch_acc:.1f}%")

---
## 10. 三范式对比分析 <a id='10'></a>

In [ ]:
# 汇总对比
print("\n" + "=" * 70)
print("三范式 + Baseline 对比总结")
print("=" * 70)

results = {
    'Zero-Shot (单模板)': zero_shot_acc_single,
    'Zero-Shot (多模板)': zero_shot_acc_ensemble,
    'Linear Probe': linear_probe_acc,
    'Fine-Tune': finetune_acc,
    '从零训练 (Baseline)': scratch_acc,
}

print(f"\n{'方法':<25s} | {'准确率':>8s} | {'需要标注数据?':>12s} | {'训练参数':>12s}")
print("-" * 70)

annotations = {
    'Zero-Shot (单模板)': ('否', '0'),
    'Zero-Shot (多模板)': ('否', '0'),
    'Linear Probe': ('是 (少量)', '仅分类头'),
    'Fine-Tune': ('是', '全部'),
    '从零训练 (Baseline)': ('是', '全部 (随机初始化)'),
}

for method, acc in results.items():
    need_data, params = annotations[method]
    print(f"{method:<25s} | {acc:>7.1f}% | {need_data:>12s} | {params:>12s}")

In [ ]:
# 可视化对比
fig, ax = plt.subplots(figsize=(10, 6))

methods = list(results.keys())
accs = list(results.values())
colors = ['#4CAF50', '#66BB6A', '#2196F3', '#FF9800', '#9E9E9E']

bars = ax.barh(methods, accs, color=colors, height=0.6)
ax.set_xlabel('Test Accuracy (%)', fontsize=12)
ax.set_title('CLIP 三种使用范式对比 (CIFAR-10)', fontsize=14)

for bar, acc in zip(bars, accs):
    ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2, 
            f'{acc:.1f}%', va='center', fontsize=11, fontweight='bold')

ax.set_xlim(0, max(accs) + 10)
plt.tight_layout()
plt.show()

print("""
分析:

1. Zero-Shot: 完全不需要标注数据！准确率取决于预训练质量和文本 prompt 设计。
   多模板 ensemble 能显著提升效果。

2. Linear Probe: 冻结编码器只训练线性头。如果 CLIP 特征足够好，
   效果可以很接近 fine-tune。训练极快。

3. Fine-Tune: 通常效果最好，但需要更多数据和计算。
   有过拟合风险，需要差异化学习率。

4. 从零训练: 没有预训练的好处，需要更多数据才能达到同等效果。

💡 关键洞察:
   - CLIP 预训练特征的价值 = Fine-Tune ACC - 从零训练 ACC
   - 数据越少，预训练越重要 (few-shot 场景下差距更大)
   - Zero-Shot 是 CLIP 最 unique 的能力
""")

In [ ]:
# 特征空间可视化: t-SNE
from sklearn.manifold import TSNE

print("\n特征空间 t-SNE 可视化...")

# 用测试集的 CLIP 特征
n_vis = min(1000, len(test_features))
vis_features = test_features[:n_vis].numpy()
vis_labels = test_labels[:n_vis].numpy()

# t-SNE 降维
tsne = TSNE(n_components=2, random_state=42, perplexity=30)
features_2d = tsne.fit_transform(vis_features)

# 画图
fig, ax = plt.subplots(figsize=(10, 8))
scatter = ax.scatter(
    features_2d[:, 0], features_2d[:, 1],
    c=vis_labels, cmap='tab10', s=15, alpha=0.7
)

# 添加图例
legend_elements = [plt.Line2D([0], [0], marker='o', color='w', 
                              markerfacecolor=plt.cm.tab10(i/10), markersize=8,
                              label=CIFAR10_CLASSES[i]) 
                   for i in range(10)]
ax.legend(handles=legend_elements, loc='upper right', fontsize=9)
ax.set_title('CLIP Image Features t-SNE (CIFAR-10)', fontsize=14)
ax.set_xlabel('t-SNE dim 1')
ax.set_ylabel('t-SNE dim 2')

plt.tight_layout()
plt.show()
print("💡 聚类越明显 → CLIP 特征质量越好 → Linear Probe 效果越好")

---
## 11. 使用 OpenAI 预训练 CLIP (可选) <a id='11'></a>

如果安装了 `open_clip_torch`，可以直接用预训练的 CLIP 模型做 zero-shot 分类。

In [ ]:
# 使用预训练 CLIP (需要安装 open_clip_torch)
# 如果无法安装，可以跳过这部分

try:
    import open_clip
    
    print("加载预训练 CLIP (ViT-B/32)...")
    pretrained_model, _, preprocess = open_clip.create_model_and_transforms(
        'ViT-B-32', pretrained='laion2b_s34b_b79k'
    )
    pretrained_tokenizer = open_clip.get_tokenizer('ViT-B-32')
    pretrained_model = pretrained_model.to(device).eval()
    
    # 重新加载测试数据 (用 CLIP 的预处理)
    test_dataset_clip = torchvision.datasets.CIFAR10(
        root='./data', train=False, download=True, transform=preprocess
    )
    test_loader_clip = DataLoader(
        Subset(test_dataset_clip, range(2000)), 
        batch_size=256, shuffle=False
    )
    
    # Zero-shot with pretrained CLIP
    text_inputs = pretrained_tokenizer(
        [f"a photo of a {c}" for c in CIFAR10_CLASSES]
    ).to(device)
    
    with torch.no_grad():
        text_features = pretrained_model.encode_text(text_inputs)
        text_features = F.normalize(text_features, dim=-1)
    
    correct = 0
    total = 0
    
    with torch.no_grad():
        for images, labels in test_loader_clip:
            images = images.to(device)
            image_features = pretrained_model.encode_image(images)
            image_features = F.normalize(image_features, dim=-1)
            
            similarity = image_features @ text_features.T
            preds = similarity.argmax(dim=1)
            correct += (preds == labels.to(device)).sum().item()
            total += len(labels)
    
    pretrained_acc = correct / total * 100
    print(f"\n预训练 CLIP (ViT-B/32) Zero-Shot 准确率: {pretrained_acc:.1f}%")
    print(f"我们训练的 CLIP Zero-Shot 准确率: {zero_shot_acc_ensemble:.1f}%")
    print(f"\n💡 差距来自: 数据量 (400M vs 10K)、模型大小 (ViT-B vs 小 CNN)、训练时间")
    
except ImportError:
    print("open_clip_torch 未安装，跳过预训练 CLIP 对比。")
    print("可以运行: pip install open_clip_torch")
    print("\n参考数据: 预训练 CLIP ViT-B/32 在 CIFAR-10 上 zero-shot 准确率约 89-92%")

---
## 12. 思考题 & 总结 <a id='12'></a>

### 今日核心知识点

1. **对比学习**: 拉近正样本对，推远负样本对
2. **InfoNCE Loss**: N 分类交叉熵，本质是从 N 个候选中找正确匹配
3. **温度参数 τ**: 控制 softmax 尖锐度，CLIP 中可学习
4. **CLIP 双塔架构**: Image Encoder + Text Encoder + 共享投影空间
5. **三种使用范式**:
   - Zero-Shot: 无需标注，直接用文本 prompt 分类
   - Linear Probe: 冻结编码器，只训练线性头
   - Fine-Tune: 解冻全部参数，差异化学习率

### 与后续课程的关联

| 今日内容 | 后续关联 |
|---------|--------|
| CLIP Score | → W2 Day5 Diffusion 评估指标 |
| 对比学习特征 | → W2 Day6 合成数据筛选 |
| Zero-Shot | → W2 Day3 评估方法论 |
| Image Encoder | → W1 Day6 ViT/ResNet backbone |

### 面试高频思考题

1. **InfoNCE 和交叉熵有什么关系？** (提示: N 分类)
2. **CLIP 的 batch size 为什么要很大？** (提示: 负样本数量)
3. **温度参数从 0.07 变到 0.01 会怎样？** (提示: 硬/软分配)
4. **CLIP 为什么能 zero-shot？传统 CNN 为什么不能？** (提示: 对齐空间)
5. **Linear Probe 效果好说明什么？效果差又说明什么？** (提示: 特征质量)
6. **CLIP 的局限性？** (提示: 细粒度、计数、空间关系)

### 明日预习: 损失函数 + 评估指标
- CE / Focal / Triplet / InfoNCE 的直觉对比
- P / R / F1 / mAP / FID / CLIP Score

In [ ]:
print("\n" + "=" * 60)
print("W2 Day2 完成! 🎉")
print("=" * 60)
print("""
今日成果:
  ✅ 手写 InfoNCE Loss (含对称版本)
  ✅ 温度参数实验 (loss / 梯度 / 准确率 vs τ)
  ✅ 手写完整 CLIP 架构 (Image Encoder + Text Encoder + 可学习温度)
  ✅ 在 CIFAR-10 上训练 CLIP
  ✅ Zero-Shot 分类 (单模板 + 多模板 ensemble)
  ✅ Linear Probe (冻结编码器)
  ✅ Fine-Tune (差异化学习率)
  ✅ 四种方法横向对比 + t-SNE 可视化
  ✅ (可选) 对比 OpenAI 预训练 CLIP
""")